# Day 07 — Generators & Async Generators for Streaming

**Phase 1 · Module 1: LangGraph Streaming** · ~30 min

### What I practice today
- [ ] Understand a **generator** — a function that `yield`s values **one at a time**, lazily
- [ ] See why streaming needs this: emit token chunks **as they arrive**, not all at the end
- [ ] Compose streams with **`yield from`**
- [ ] Write an **async generator** (`async def` + `yield`) consumed with **`async for`**
- [ ] Bridge: this is exactly the shape LangGraph's `.astream()` / `.astream_events()` use


## 1. The problem: return gives you *everything at once*

A normal function that builds a response holds the whole thing, then hands it back in one lump. For a chat model that's the worst UX: the user stares at a blank screen for 4 seconds, then the full answer appears at once. We want each **token** the moment it's ready.

Here's the *eager* version — it assembles the full string before you see a single character:

In [1]:
# Eager: nothing is visible until the ENTIRE answer is built.
def full_answer(prompt: str) -> str:
    tokens = ['Your', ' balance', ' is', ' \u00a31,240', '.']
    result = ''
    for t in tokens:
        result += t          # imagine each token costs 300ms of model time
    return result            # caller waits for ALL of it

print(full_answer('what is my balance?'))

Your balance is £1,240.


☝ The caller gets the answer only after **every** token exists. Fine for 5 tokens; painful for a 500-token model reply. We need to hand back tokens **as they are produced**.

## 2. The fix: `yield` — a function that pauses and resumes

Swap `return` for **`yield`** and the function becomes a **generator**. Each `yield` hands **one** value back to the caller and **freezes** the function right there; the next request resumes it from that exact line. Nothing is computed until asked for — that's **lazy** evaluation.

In [2]:
# Lazy: each token leaves the function the instant it's ready.
def stream_answer(prompt: str):
    tokens = ['Your', ' balance', ' is', ' \u00a31,240', '.']
    for t in tokens:
        yield t              # hand ONE token out, then pause here

gen = stream_answer('what is my balance?')
print(type(gen).__name__)    # 'generator' - calling it runs NO body yet

for chunk in stream_answer('what is my balance?'):
    print(repr(chunk))       # arrives one at a time

generator
'Your'
' balance'
' is'
' £1,240'
'.'


☝ Calling `stream_answer(...)` runs **none** of the body — it returns a *generator object*. The code only advances when something pulls from it (the `for` loop). Each pull runs up to the next `yield`, then pauses. That pause-and-resume is the whole mechanism behind token streaming.

## 3. Look under the hood — `next()` and `StopIteration`

A `for` loop is sugar. Underneath, Python calls **`next()`** on the generator to get each value, and catches a **`StopIteration`** exception when the generator is exhausted. Driving it by hand makes the pause/resume visible:

In [3]:
gen = stream_answer('hi')
print(next(gen))     # runs body up to first  yield -> 'Your'
print(next(gen))     # RESUMES, runs to next  yield -> ' balance'
print(next(gen))     # ' is'

# ...drain the rest, then one more next() raises StopIteration
for _ in gen:
    pass
try:
    next(gen)
except StopIteration:
    print('generator exhausted -> StopIteration')

Your
 balance
 is
generator exhausted -> StopIteration


☝ Each `next()` **resumes** the frozen function until the next `yield`, then freezes again. When the body finally returns, Python raises `StopIteration` — the `for` loop catches that quietly and stops. A generator is single-use: once exhausted, it's done.

## 4. State survives across yields — accumulate as you stream

Because the function's frame is *paused*, local variables live on between yields. That lets a streamer emit a chunk **and** keep a running total — exactly what a UI needs: show each token, and know the full text so far.

In [4]:
def stream_with_running_total(tokens):
    sofar = ''
    for t in tokens:
        sofar += t                       # local state persists across yields
        yield t, sofar                   # emit the chunk AND the text so far

toks = ['Fraud', ' check', ':', ' PASS']
for chunk, sofar in stream_with_running_total(toks):
    print(f'chunk={chunk!r:12}  full so far={sofar!r}')

chunk='Fraud'       full so far='Fraud'
chunk=' check'      full so far='Fraud check'
chunk=':'           full so far='Fraud check:'
chunk=' PASS'       full so far='Fraud check: PASS'


☝ `sofar` isn't reset between iterations — the paused frame keeps it alive. This is how a streaming terminal UI can print each new token *and* always have the complete message for logging or a final commit.

## 5. Compose streams with `yield from`

Real agents stream from **several** sources — a planner's thoughts, then a tool's output, then the final answer. **`yield from another_gen`** delegates: it re-yields every value of the inner generator, so callers see one flat stream.

In [5]:
def think(goal):
    yield '[thinking] parsing goal...\n'
    yield f'[thinking] goal = {goal!r}\n'

def answer():
    for t in ['Approved', '.', ' Limit', ' set', ' to', ' \u00a350k', '.']:
        yield t

def agent_stream(goal):
    yield from think(goal)      # splice in every chunk from think()
    yield '\n[answer] '
    yield from answer()         # then every chunk from answer()

for chunk in agent_stream('raise credit limit'):
    print(chunk, end='')

[thinking] parsing goal...
[thinking] goal = 'raise credit limit'

[answer] Approved. Limit set to £50k.

☝ `yield from` flattens nested generators into **one** stream — the caller never knows there were two sources. That's how LangGraph merges node-by-node events into a single sequence you can loop over once.

## 6. Why *async*? Because each token arrives over the **network**

A real model streams tokens across an HTTP connection — between chunks you're **waiting on I/O**, and waiting should `await` so the event loop isn't blocked (Day 03). An **async generator** is the answer: `async def` + `yield`, and each step can `await`. You consume it with **`async for`**.

In [6]:
import asyncio

async def astream_answer(prompt: str):
    tokens = ['Your', ' balance', ' is', ' \u00a31,240', '.']
    for t in tokens:
        await asyncio.sleep(0.05)     # model 'thinking' between tokens = real I/O
        yield t                       # async generator: async def + yield

async def main():
    async for chunk in astream_answer('balance?'):   # note: async for
        print(chunk, end='', flush=True)
    print()

await main()      # notebooks have a running loop, so await directly

Your balance is £1,240.


☝ The `await asyncio.sleep(...)` **between** yields is the network gap of a real model. During each gap the event loop is free to do other work (handle another request, run another node). `async for` is just the async twin of the `for` loop from §2 — same pause/resume, now non-blocking.

## 7. The payoff: first token is **instant**

Streaming's win is **time-to-first-token**. Eager makes the user wait for the sum of all token delays before *anything* shows. Streaming shows token #1 after just *one* delay. Time both to see it:

In [7]:
import time

async def eager(tokens):
    out = ''
    for t in tokens:
        await asyncio.sleep(0.05)
        out += t
    return out                          # nothing visible until here

async def demo():
    toks = ['A'] * 8
    t0 = time.perf_counter()
    await eager(toks)
    print(f'eager   : first output after {time.perf_counter()-t0:.2f}s')

    t0 = time.perf_counter()
    async for i, _ in _enumerate(astream_answer('x')):
        if i == 0:
            print(f'streaming: first token after {time.perf_counter()-t0:.2f}s')

async def _enumerate(agen):            # async enumerate helper
    i = 0
    async for x in agen:
        yield i, x
        i += 1

await demo()

eager   : first output after 0.41s
streaming: first token after 0.05s


☝ Same total time to finish — but the streamed version puts the **first** token on screen after a single hop instead of eight. On a 500-token model reply that's the difference between a 100ms and a 5s wait before the user sees life. **That is why every agent UI streams.**

## 8. Tie-in — this is exactly LangGraph's streaming shape

In today's LangGraph notebook you'll call `graph.astream(...)` and `graph.astream_events(...)`. Both are **async generators**: you drive them with `async for`, and each yield is one event (a token, a node start, a tool result). The generator protocol you just learned *is* the API — nothing new to memorise, only new event shapes to read.

Here's a mock of that exact call shape so the bridge is obvious:

In [8]:
async def graph_astream(goal):
    # mimics LangGraph: yields (node_name, chunk) events as the graph runs
    yield ('planner', 'draft plan\n')
    for t in ['Summar', 'ise', ' fraud', ' policy']:
        await asyncio.sleep(0.03)
        yield ('executor', t)          # token-level events from a node
    yield ('replan', ' [done]')

async def run():
    async for node, chunk in graph_astream('summarise policies'):
        print(f'{node:9} | {chunk!r}')

await run()

planner   | 'draft plan\n'
executor  | 'Summar'
executor  | 'ise'
executor  | ' fraud'
executor  | ' policy'
replan    | ' [done]'


☝ Swap `graph_astream` for a real `graph.astream(..., stream_mode='messages')` and the consumer loop is **identical**. You already know how to read the stream — the LangGraph notebook just wires a real graph behind it.

## 9. Recap + checklist

| Tool | Why it was used here |
|------|----------------------|
| **`yield`** | turn a function into a generator — hand back one value at a time, lazily |
| **generator object** | calling the fn runs no body; work happens only when pulled (`for` / `next`) |
| **`next()` / `StopIteration`** | the protocol under `for`: resume to next `yield`, stop when exhausted |
| **paused frame / locals** | state survives across yields — emit a chunk *and* keep a running total |
| **`yield from`** | flatten several generators into one stream (planner → tool → answer) |
| **`async def` + `yield`** | async generator — `await` network I/O *between* tokens |
| **`async for`** | consume an async generator without blocking the event loop |
| **time-to-first-token** | the reason to stream: first token instant vs. waiting for the whole reply |
